# TensorFlow Logistic Regression + FGM Pipeline

This notebook keeps the same overall adversarial pipeline, but uses a **TensorFlow logistic regression model** wrapped with ART's `TensorFlowV2Classifier`.

This is still logistic regression in the model sense:
- **no hidden layers**
- one linear output layer
- softmax over 2 classes

Pipeline steps:

1. Train a clean TensorFlow logistic regression model  
2. Evaluate the clean model on clean test data  
3. Initialize the FGM attack  
4. Generate adversarial train and test samples  
5. Keep adversarial labels aligned with the original ground-truth labels  
6. Build combined clean+adversarial train and test sets  
7. Adversarially retrain a fresh TensorFlow logistic regression model with ART's `AdversarialTrainer`  
8. Evaluate both the clean model and the adversarially trained model on:
   - clean test data
   - adversarial test data
   - combined clean+adversarial test data
9. Run the same consistency gate used in the NN notebook


In [ ]:
# If needed, install once:
# !pip install tensorflow adversarial-robustness-toolbox scikit-learn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DEFAULT_DATA_PATH = Path(r"CSVs\newDataset.csv")
RUNS_DIR = Path(r"StandardizedRuns")
RUN_GLOB = "NeuralNet_train_*.csv"

# Optional environment overrides:
# - TF_LR_RUN_PATH
# - MODEL_RUN_PATH
ENV_RUN_PATH = os.environ.get("TF_LR_RUN_PATH") or os.environ.get("MODEL_RUN_PATH")

LABEL_COL = "anomaly"
DROP_COLS = {LABEL_COL, "segment", "train", "sampling"}

TEST_SIZE = 0.75
BATCH_SIZE = 128
NB_EPOCHS = 20
LR = 1e-3
FGM_EPS = 0.10
ADV_RATIO = 0.55
SAVE_MODELS = True

ARTIFACT_DIR = Path(r"Results\TensorFlowLogisticRegressionResults")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    X = df[feature_cols].to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler


def resolve_run_path():
    if ENV_RUN_PATH:
        path = Path(ENV_RUN_PATH)
        if not path.exists():
            raise FileNotFoundError(f"Run file not found: {path}")
        return path

    candidates = sorted(RUNS_DIR.glob(RUN_GLOB))
    if candidates:
        return candidates[-1]

    if DEFAULT_DATA_PATH.exists():
        return DEFAULT_DATA_PATH

    raise FileNotFoundError(
        "Could not find a dataset automatically. Set TF_LR_RUN_PATH or MODEL_RUN_PATH, "
        "or place the CSV at CSVs\newDataset.csv."
    )


RUN_PATH = resolve_run_path()
X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(str(RUN_PATH))


In [ ]:
def make_tf_lr_model(d_in: int):
    # Logistic regression in neural-network form: one Dense layer, no hidden layers.
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(d_in,)),
        tf.keras.layers.Dense(2, activation="softmax")
    ])
    return model


def make_art_classifier(d_in: int, lr: float = LR):
    model = make_tf_lr_model(d_in)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

    @tf.function
    def train_step(model, images, labels):
        with tf.GradientTape() as tape:
            predictions = model(images, training=True)
            loss = loss_object(labels, predictions)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    classifier = TensorFlowV2Classifier(
        model=model,
        nb_classes=2,
        input_shape=(d_in,),
        loss_object=loss_object,
        train_step=train_step,
        clip_values=(0.0, 1.0),
    )
    return classifier


def predict_labels(art_clf: TensorFlowV2Classifier, X: np.ndarray):
    probs = art_clf.predict(X)
    return np.argmax(probs, axis=1)


def eval_from_predictions(y_true, y_pred, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n{name}")
    print(f"acc: {acc:.6f}")
    print(f"f1 : {f1:.6f}")
    print("confusion_matrix:")
    print(cm)
    print("classification_report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "name": name,
        "acc": acc,
        "f1": f1,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }


def eval_classifier(art_clf: TensorFlowV2Classifier, X: np.ndarray, y_true: np.ndarray, name: str):
    y_pred = predict_labels(art_clf, X)
    metrics = eval_from_predictions(y_true, y_pred, name)
    return metrics, y_pred


def save_art_model(art_clf: TensorFlowV2Classifier, save_path: Path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    art_clf.model.save(save_path)
    print(f"Saved model: {save_path}")


In [ ]:
# Step 1: clean.fit(X_train, y_train) -> trained clean model
print('Training clean TensorFlow logistic regression model with:', {
    'batch_size': BATCH_SIZE,
    'nb_epochs': NB_EPOCHS,
    'learning_rate': LR,
    'fgm_eps': FGM_EPS,
    'adv_ratio': ADV_RATIO,
    'random_state': SEED,
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

if SAVE_MODELS:
    save_art_model(art_clean, ARTIFACT_DIR / "tf_lr_clean_model.keras")


In [ ]:
# Step 2: initialize FGM and generate adversarial samples
fgm = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)

X_train_adv = fgm.generate(x=X_train)
X_test_adv = fgm.generate(x=X_test)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# Optional predictions on adversarial inputs
_y_train_adv_pred = predict_labels(art_clean, X_train_adv)
_y_test_adv_pred = predict_labels(art_clean, X_test_adv)

# Keep original ground-truth labels aligned with adversarial samples
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

# Build combined clean + adversarial datasets
X_train_combined = np.concatenate([X_train, X_train_adv], axis=0).astype(np.float32)
y_train_combined = np.concatenate([y_train, y_train_adv], axis=0).astype(np.int64)

X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)

print("Combined train set:", X_train_combined.shape, y_train_combined.shape)
print("Combined test set :", X_test_combined.shape, y_test_combined.shape)


In [ ]:
# Evaluate performance of the clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)


In [ ]:
# Retrain using ART's AdversarialTrainer
art_adv = make_art_classifier(d_in=X_train.shape[1])

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=fgm,
    ratio=ADV_RATIO,
)

adv_trainer.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

if SAVE_MODELS:
    save_art_model(art_adv, ARTIFACT_DIR / "tf_lr_adversarial_trained_model.keras")


In [ ]:
# Test using X_test_adv and the combined clean+adv test set
adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)


In [ ]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

summary_df


In [ ]:
# Save metrics
out_csv = ARTIFACT_DIR / "tf_lr_fgm_evasion_pipeline.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


## Consistency Gate

This keeps the same simple ADEPTRS-style consistency gate:

- **Anomaly** if both models flag anomaly
- **AttackFlag** if only the robust model flags anomaly
- **Nominal** otherwise

The clean TensorFlow logistic regression model acts as the **Nominal Specialist** and the adversarially trained TensorFlow logistic regression model acts as the **Robust Guardian**.


In [ ]:
# Simple consistency gate
TA = 0.90   # nominal specialist anomaly threshold
TB = 0.60   # robust guardian anomaly threshold

def get_anomaly_probs(art_clf: TensorFlowV2Classifier, X: np.ndarray):
    probs = art_clf.predict(X)
    return probs[:, 1]


def consistency_gate(art_nominal: TensorFlowV2Classifier, art_robust: TensorFlowV2Classifier, X: np.ndarray, ta: float = TA, tb: float = TB):
    p_nominal_anom = get_anomaly_probs(art_nominal, X)
    p_robust_anom = get_anomaly_probs(art_robust, X)

    nominal_flags = (p_nominal_anom >= ta).astype(int)
    robust_flags = (p_robust_anom >= tb).astype(int)

    gate_labels = []
    for m1, m2 in zip(nominal_flags, robust_flags):
        if m1 == 1 and m2 == 1:
            gate_labels.append("Anomaly")
        elif m1 == 0 and m2 == 1:
            gate_labels.append("AttackFlag")
        else:
            gate_labels.append("Nominal")

    gate_df = pd.DataFrame({
        "p_nominal_anom": p_nominal_anom,
        "p_robust_anom": p_robust_anom,
        "nominal_flag": nominal_flags,
        "robust_flag": robust_flags,
        "gate_label": gate_labels,
    })

    return gate_df


def summarize_gate(gate_df: pd.DataFrame, y_true: np.ndarray, name: str):
    mapped_pred = gate_df["gate_label"].map({"Nominal": 0, "Anomaly": 1, "AttackFlag": 1}).to_numpy()
    acc = accuracy_score(y_true, mapped_pred)
    f1 = f1_score(y_true, mapped_pred, zero_division=0)
    cm = confusion_matrix(y_true, mapped_pred)

    print(f"\n{name}")
    print(gate_df["gate_label"].value_counts(dropna=False))
    print(f"acc: {acc:.6f}")
    print(f"f1 : {f1:.6f}")
    print("confusion_matrix:")
    print(cm)

    return {
        "name": name,
        "acc": acc,
        "f1": f1,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "n_nominal": int((gate_df["gate_label"] == "Nominal").sum()),
        "n_anomaly": int((gate_df["gate_label"] == "Anomaly").sum()),
        "n_attackflag": int((gate_df["gate_label"] == "AttackFlag").sum()),
    }


gate_clean_df = consistency_gate(art_clean, art_adv, X_test)
gate_adv_df = consistency_gate(art_clean, art_adv, X_test_adv)
gate_combined_df = consistency_gate(art_clean, art_adv, X_test_combined)

gate_summary_df = pd.DataFrame([
    summarize_gate(gate_clean_df, y_test, "gate_on_clean_test"),
    summarize_gate(gate_adv_df, y_test_adv, "gate_on_adv_test"),
    summarize_gate(gate_combined_df, y_test_combined, "gate_on_combined_test"),
])

gate_summary_df


In [ ]:
# Save gate outputs
gate_out_csv = ARTIFACT_DIR / "tf_lr_consistency_gate_summary.csv"
gate_out_csv.parent.mkdir(parents=True, exist_ok=True)
gate_summary_df.to_csv(gate_out_csv, index=False)

gate_clean_df.to_csv(gate_out_csv.parent / "tf_lr_gate_clean_predictions.csv", index=False)
gate_adv_df.to_csv(gate_out_csv.parent / "tf_lr_gate_adv_predictions.csv", index=False)
gate_combined_df.to_csv(gate_out_csv.parent / "tf_lr_gate_combined_predictions.csv", index=False)

print(f"Saved: {gate_out_csv}")
